[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/05_finetune_and_compare.ipynb)

# Step 5 — Fine-Tune and Compare (Optional Demo)

LoRA fine-tune a small model on the grounded synthetic corpus, then re-run the Step 1 test set.

## Learning objectives
- Prepare an instruction dataset for SFT
- Fine-tune with TRL + PEFT (4-bit LoRA)
- Compare baseline vs fine-tuned scores by failure mode

## Prerequisit:
1. Generated synthetic_train.jsonl and test_set.jsonl files
2. Install SFT dependencies:
    ``` bash
    uv sync --group text-sft
    ```
3. Set ``RUN_SFT`` to 1 to run GPU LoRA fine-tuning in .env (needs CUDA; not via Ollama)

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_SCORES_PATH,
    COMPARISON_REPORT_PATH,
    FINETUNED_PREDICTIONS_PATH,
    SYNTHETIC_TRAIN_PATH,
    TEST_SET_PATH,
    QASample,
    build_sft_dataset,
    compare_summaries,
    create_judge_client,
    create_small_model_client,
    load_typed_jsonl,
    qa_samples_to_messages,
    read_json,
    run_inference,
    save_baseline_results,
    score_predictions,
    train_lora_sft,
    use_repo_root,
    write_json,
)
from aieng.syn_data.text.sft import PeftInferenceClient
from dotenv import load_dotenv


load_dotenv()
ROOT = use_repo_root(Path("."))

MODELS_DIR = ROOT / "implementations" / "qa_text_generation" / "models" / "lora_adapter"
BASE_MODEL = os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-3B-Instruct")
RUN_SFT = os.getenv("RUN_SFT", "1") == "1"

2026-06-11 17:02:02,760 INFO root: AI Engineering synthetic data utilities 

 Logging configured.


In [2]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Load training and test data

In [3]:
train_samples = load_typed_jsonl(SYNTHETIC_TRAIN_PATH, QASample.from_dict)
test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
print(len(train_samples), len(test_samples))

7 6


## 2. LoRA SFT with TRL

Suggested base model: `Qwen/Qwen2.5-3B-Instruct` with 4-bit quantization.

In [4]:
sft_dataset = build_sft_dataset(train_samples)
print(sft_dataset)
print("Example training row:")
qa_samples_to_messages(train_samples[:1])[0]

if RUN_SFT:
    adapter_path = train_lora_sft(
        train_samples,
        MODELS_DIR,
        base_model=BASE_MODEL,
        num_train_epochs=1.0,
    )
    print(f"Saved LoRA adapter to {adapter_path}")
else:
    print(
        "SFT skipped (set RUN_SFT=1 and run on a CUDA machine to fine-tune). "
        "Fine-tuned evaluation will reuse the small model client.",
    )

/Users/royajavadi/projects/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['messages'],
    num_rows: 7
})
Example training row:


RuntimeError: CUDA is required for LoRA fine-tuning. Set RUN_SFT=1 only on a GPU machine.

## 3. Re-evaluate on the held-out test set

In [ ]:
judge = create_judge_client()

if RUN_SFT and MODELS_DIR.exists():
    finetuned_client = PeftInferenceClient(MODELS_DIR, BASE_MODEL)
else:
    finetuned_client = create_small_model_client()

finetuned_predictions = run_inference(finetuned_client, test_samples)
finetuned_scores = score_predictions(judge, test_samples, finetuned_predictions)

finetuned_summary = save_baseline_results(
    finetuned_predictions,
    finetuned_scores,
    test_samples,
    predictions_path=FINETUNED_PREDICTIONS_PATH,
    scores_path=COMPARISON_REPORT_PATH.parent / "finetuned_scores.json",
)
finetuned_summary

## 4. Compare before vs after

In [ ]:
baseline_report = read_json(BASELINE_SCORES_PATH)
comparison = {
    "baseline": baseline_report["overall"],
    "finetuned": finetuned_summary["overall"],
    "delta": compare_summaries(
        baseline_report["overall"],
        finetuned_summary["overall"],
    ),
    "by_failure_mode": {
        "baseline": baseline_report.get("by_failure_mode", {}),
        "finetuned": finetuned_summary.get("by_failure_mode", {}),
    },
    "run_sft": RUN_SFT,
}
write_json(COMPARISON_REPORT_PATH, comparison)
comparison